# 11 · Gridded data at scale

*Why the shape of the chunks decides what every read costs.*

Everything so far has been tables of stations. Most weather data isn't. Model output, reanalysis and satellite products are **grids**: one value per cell, per time, sometimes per height and per ensemble member. ERA5 alone is petabytes. Nobody loads a grid like that into memory. You open it lazily, slice what you need, and only the pieces holding your slice move.

How fast that is depends almost entirely on how the array was cut up when it was written. This notebook shows that with a real, mid-sized array: every hour of HRRR's 2 m temperature over California for 2021–2024, which notebook 03 kept as a side product. One variable is 35,064 hours × 458 × 420 cells, about 27 GB uncompressed. That's too big to treat casually on a laptop and small enough to experiment with.

Three ideas carry it:

- **Zarr** stores an n-dimensional array as a grid of compressed **chunks**, one object each, plus a little JSON. It's the format behind WeatherBench 2, ARCO-ERA5 and ECMWF's Anemoi training data.
- **Chunk shape must match the reader.** A training job reads whole maps. A climate study reads long series at a point. One layout can't be fast for both.
- **Rechunking** converts one layout into another. Done naively it needs the whole array in memory. **dask** and **xarray-beam** do it in steps under a memory limit, the second on any Beam runner up to a cluster.

**Read first:** Zarr's [Quickstart](https://zarr.readthedocs.io/en/stable/quickstart.html) and [user guide on arrays, chunks, compression and sharding](https://zarr.readthedocs.io/en/stable/user-guide/arrays.html); xarray's [Reading and writing Zarr](https://docs.xarray.dev/en/stable/user-guide/io.html#zarr); the [xarray-beam README](https://github.com/google/xarray-beam) and ["Why xarray-beam"](https://xarray-beam.readthedocs.io/en/latest/why-xarray-beam.html). About 45 minutes.

In [ ]:
# Setup: identical in every notebook. Data lives in ../data, which git ignores.
import os
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
os.makedirs(DATA_DIR, exist_ok=True)
print("Data directory:", DATA_DIR)

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
from wxlab import gridded as gd

plt.rcParams["figure.dpi"] = 80
GRID = Path(DATA_DIR) / "reference" / "hrrr_grid_ca"
ZARR = Path(DATA_DIR) / "work" / "zarr"
TRAIN, ANALYSIS, ANALYSIS_BEAM = ZARR / "t2m_training.zarr", ZARR / "t2m_analysis.zarr", ZARR / "t2m_analysis_beam.zarr"
ZARR.mkdir(parents=True, exist_ok=True)

## Step 1 · One labelled cube

The day files are plain NumPy arrays. xarray gives them names and coordinates: a `time` axis, `y` and `x` grid indices, and 2D `latitude` and `longitude` because HRRR's grid is a Lambert conformal projection and not a latitude–longitude grid. CF attributes say what the numbers are.

`gd.template` builds that dataset lazily: full shape, coordinates, attributes, and no data yet.

In [ ]:
tpl = gd.template(GRID)
print(tpl)
print(f"{tpl.air_temperature.nbytes / 1e9:.1f} GB uncompressed")

## Step 2 · Write the training layout

Training reads whole maps, so the first layout uses one chunk per day: 24 hours × the whole state, about 18 MB uncompressed per chunk. Writing happens in units of one month. The template is written first (metadata only), then each month fills its own region of the time axis. A log of finished months makes the job resumable, the same idea as the manifest in notebook 07.

In [ ]:
gd.write_training_layout(GRID, TRAIN, chunks=(24, None, None))
train = xr.open_zarr(TRAIN)
print(train.air_temperature.encoding["chunks"], gd.store_stats(TRAIN))

*Draft note: report objects and GB on disk and the compression ratio against 27 GB. Temperature fields are smooth, so expect a good ratio. Mention that opening the store read only metadata.*

## Step 3 · Rechunk to the analysis layout with dask

The analysis layout cuts the cube into long thin columns: every hour × a 16 × 16 tile, about 36 MB per chunk. Turning day-shaped chunks into column-shaped ones means every output chunk needs a little piece of every input chunk. That's the array version of a shuffle. dask plans it as a graph of small copy tasks and runs it with bounded memory.

In [ ]:
from dask.diagnostics import ProgressBar

target = {"time": -1, "y": 16, "x": 16}
if not ANALYSIS.exists():
    with ProgressBar():
        train.chunk(target).to_zarr(ANALYSIS, zarr_format=3,
                                    encoding={"air_temperature": {"chunks": (train.sizes["time"], 16, 16)}})
analysis = xr.open_zarr(ANALYSIS)
print(analysis.air_temperature.encoding["chunks"], gd.store_stats(ANALYSIS))

*Draft note: report wall time and peak memory (watch Activity Monitor or use `dask.diagnostics.ResourceProfiler`). If memory runs out, rechunk in two steps through an intermediate layout (for example 24 × 7 days by 64 × 64 tiles), which is what dedicated rechunking tools do.*

## Step 4 · Measure what each read costs

Two typical questions, asked of both layouts:

- **one map**: the whole state at one hour, what a training step reads;
- **one place, four years**: every hour at one cell, what a climate study reads.

In [ ]:
iy, ix = 230, 210                                    # a cell near the middle of the crop
rows = []
for name, ds in [("training (day × state)", train), ("analysis (all hours × 16²)", analysis)]:
    v = ds.air_temperature
    rows.append({"layout": name,
                 "one map: chunks": gd.chunks_touched(v, time=1000),
                 "one map: s": gd.timed(lambda: v.isel(time=1000).values),
                 "one place: chunks": gd.chunks_touched(v, y=iy, x=ix),
                 "one place: s": gd.timed(lambda: v.isel(y=iy, x=ix).values, repeat=1)})
pd.DataFrame(rows).set_index("layout").round(3)

*Draft note: expect "one map" to touch 1 chunk in the training layout and about 27 × 27 = 750 in the analysis layout, and "one place" the reverse (1,461 chunks against 1). Report the times and the ratio. This is the table the project page's petabyte section refers to.*

The general rule: a read costs roughly the number of chunks it touches times the cost of one chunk. Chunks that are too small make every read a flood of requests, and chunks that are too big make small reads drag in bytes they don't need. On cloud object storage the usual target is about 10–100 MB per chunk. When two kinds of readers matter, big producers keep two copies, as ECMWF does with ARCO ERA5.

## Step 5 · The same rechunk with xarray-beam

xarray-beam expresses the rechunk as a Beam pipeline over `(key, chunk)` pairs. `DatasetToChunks` reads the source in its own chunks, `Rechunk` splits and regroups them under a memory budget, and `ChunksToZarr` writes the result. On the DirectRunner it's slower than dask. The point is that the same pipeline runs on Dataflow across hundreds of machines, which is how WeatherBench 2 processes ERA5.

In [ ]:
import apache_beam as beam
import xarray_beam as xbeam

src_chunks = {"time": 24, "y": -1, "x": -1}
dst_chunks = {"time": -1, "y": 16, "x": 16}
source = xr.open_zarr(TRAIN, chunks=None)
tmpl = xbeam.make_template(source)
if not ANALYSIS_BEAM.exists():
    with beam.Pipeline() as p:
        (p | xbeam.DatasetToChunks(source, src_chunks)
           | xbeam.Rechunk(source.sizes, src_chunks, dst_chunks, itemsize=4, max_mem=2**30)
           | xbeam.ChunksToZarr(str(ANALYSIS_BEAM), tmpl, dst_chunks))
check = xr.open_zarr(ANALYSIS_BEAM).air_temperature.isel(y=iy, x=ix).values
print("identical to the dask result:", np.allclose(check, analysis.air_temperature.isel(y=iy, x=ix).values, equal_nan=True))

## Step 6 · A computation over the whole cube

Last, something useful: the mean temperature for each hour of the day at every cell, over four years. That's a 24 × 458 × 420 result from 27 GB of input, computed lazily so only a few chunks are in memory at a time. It runs on the training layout because it needs whole maps in time order.

In [ ]:
with ProgressBar():
    diurnal = train.air_temperature.groupby("time.hour").mean().compute() - 273.15
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, h, title in [(axes[0], 12, "12 UTC (4–5 am local)"), (axes[1], 0, "00 UTC (4–5 pm local)")]:
    m = ax.pcolormesh(train.longitude, train.latitude, diurnal.sel(hour=h), cmap="RdYlBu_r", vmin=-5, vmax=35)
    ax.set_title(f"Mean 2 m temperature, {title}", fontsize=10)
fig.colorbar(m, ax=axes, label="°C", shrink=0.85)

*Draft note: describe the map once run. Expect the Central Valley's big day–night swing, the coast's small one, and the Sierra cold at both times.*

## How this scales to petabytes

Nothing here depended on the laptop. On a petabyte the same steps hold:

- **Write once in units** keyed by time, into a store whose shape is declared up front, so many writers can fill regions in parallel and a failed unit is just rerun.
- **Choose chunks for the main reader** and keep a second copy for the other one, made once with a distributed rechunk (xarray-beam on Dataflow, or dask on a cluster).
- **Read lazily, reduce early, compute near the data.** A notebook opens the store, describes the slice, and lets a cluster in the same cloud region do the work.
- **Avoid copying at all** where possible. Virtual Zarr (VirtualiZarr, Kerchunk) can expose existing GRIB or NetCDF archives as Zarr without rewriting them, and Icechunk adds versions so a model can name the exact data it trained on.

## What surprised me

*Draft: fill in after running.*

## Check yourself

<details><summary>Why is rechunking from day-chunks to column-chunks expensive?</summary>
Every output chunk needs a slice of every input chunk along time, so all the data has to be read and regrouped. It's the array version of a shuffle.
</details>

<details><summary>A model trains on random single hours from the whole period. Which layout should its data loader read?</summary>
The training layout, ideally with chunks of one hour or a few, so each sample touches one small chunk. The analysis layout would read about 750 chunks per sample.
</details>

<details><summary>Why declare the full template before writing any month?</summary>
So the store's shape and chunk grid are fixed, months can be written in any order or in parallel into their own regions, and a crashed month can be redone without touching the others.
</details>

New terms from this notebook are in [`docs/glossary.md`](../docs/glossary.md).